# 01 · Khám phá dữ liệu đã chọn cho FOX INPUT

[EDA 0.5](../docs/research/eda-0.5-data-review.md) chọn vai trò của từng nguồn. Notebook 01 đo khả năng
chuẩn bị pool INPUT đa lĩnh vực cho Prompt Guard 2, hai nhãn 0/1. Mốc 40.000 và tỷ lệ 45/45/10 là
kịch bản dự toán, giữ VI và EN cân bằng và dành 10% cho câu trộn.

| Vai trò | Nguồn | Đơn vị văn bản |
|---|---|---|
| Có nhãn nguồn | SecAI, VI–EN, GuardSpec | text |
| Cần gán nhãn | Aya, OASST2 | inputs; lượt prompter hợp lệ VI/EN |
| Dự phòng | TrustAIRLab, deepset, jackhhao | prompt hoặc text, cần rà ngôn ngữ và upstream |

Vi-Alpaca, PhoATIS và các bộ CONTEXT/OUTPUT giữ vai trò đã đề xuất trong EDA 0.5. Những giá trị khuyết nhãn
ở đây là trạng thái phân tích, dữ liệu train cuối cùng chỉ dùng 0/1. Notebook chỉ đọc raw, không xuất tập train.
Bảng mẫu chỉ hiển thị dòng nguồn và tín hiệu, tránh đưa nội dung người dùng vào notebook.

Chạy bằng kernel Python 3 với pandas, numpy, scikit-learn, pyarrow, tokenizers và langdetect.
Nếu cần, cài thư viện còn thiếu vào môi trường notebook hoặc vào `var/eda01_deps`.
Tokenizer lấy từ checkpoint PG2-86M cục bộ. Notebook không tải model hay gọi dịch vụ ngoài.

[Nhận xét và quyết định cho ELT](../docs/research/eda-01-input-handoff.md).


## Thiết lập và hàm phân tích

Các hàm đọc dữ liệu, kiểm nhãn và thống kê được định nghĩa trong notebook này. Manifest nguồn và benchmark nằm trong thư mục EDA 0.5.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
project = next(p for p in (here, *here.parents) if (p / "notebooks/01_eda_data.ipynb").is_file())
ROOT = project
AUDIT = ROOT / "notebooks/00_5_eda_data"
local_deps = ROOT / "var/eda01_deps"
if local_deps.is_dir():
    sys.path.insert(0, str(local_deps))

import bz2
import csv
import hashlib
import json
import lzma
import re
import unicodedata
from collections import Counter, defaultdict
from itertools import combinations
from math import isnan

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from langdetect import DetectorFactory, detect_langs
from langdetect.lang_detect_exception import LangDetectException
from sklearn.feature_extraction.text import HashingVectorizer
from tokenizers import Tokenizer

csv.field_size_limit(2_000_000)


PII_PATTERN = re.compile(
    r"\b[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}\b"
    r"|(?<!\d)(?:\+84[\s.-]?|0)[35789](?:[\s.-]?\d){8}(?!\d)"
)

MARKUP_PATTERN = re.compile(r"<\s*(?:script|iframe|object|embed)\b|javascript\s*:", re.I)

SECRET_PATTERN = re.compile(
    r"-----BEGIN (?:[A-Z0-9]+ )*PRIVATE KEY-----|\b(?:ghp_|github_pat_|sk_live_)[A-Za-z0-9_]{12,}"
)


def read_llmail(path):
    with path.open(encoding="utf-8") as stream:
        if stream.readline().strip() != "{":
            raise ValueError("LLMail không khớp cấu trúc object")
        parts = []
        closed = False
        for line in stream:
            if line.rstrip("\r\n") == "}":
                closed = True
                break
            if line.startswith('  "') and parts:
                yield parse_mail_item(parts)
                parts.clear()
            parts.append(line)
        if parts:
            yield parse_mail_item(parts)
        if not closed or any(line.strip() for line in stream):
            raise ValueError("LLMail có phần cuối không hợp lệ")


def parse_mail_item(parts):
    key, row = next(iter(json.loads("{" + "".join(parts).rstrip().removesuffix(",") + "}").items()))
    return {**row, "@key": key}


def records(path):
    if path.suffix == ".parquet":
        parquet = pq.ParquetFile(path, arrow_extensions_enabled=False)
        for batch in parquet.iter_batches(batch_size=2048):
            yield from batch.to_pylist()
    elif path.suffix == ".csv":
        with path.open(encoding="utf-8-sig", newline="") as stream:
            for row in csv.DictReader(stream):
                if None in row or None in row.values():
                    raise ValueError("Dòng CSV lệch schema")
                yield row
    elif path.suffix in {".jsonl", ".xz"}:
        opener = lzma.open if path.suffix == ".xz" else open
        with opener(path, "rt", encoding="utf-8-sig") as stream:
            for line in stream:
                if len(line) > 2_000_000:
                    raise ValueError("Một dòng JSONL vượt giới hạn")
                if line.strip():
                    yield json.loads(line)
    elif path.name == "labelled_unique_submissions_phase1.json":
        yield from read_llmail(path)
    elif path.suffix == ".json" and path.stat().st_size <= 65_000_000:
        with path.open(encoding="utf-8-sig") as stream:
            rows = json.load(stream)
        if not isinstance(rows, list):
            raise ValueError("File JSON phải là danh sách record")
        yield from rows
    else:
        raise ValueError("Định dạng EDA không được hỗ trợ")


def values(node, key):
    if isinstance(node, list):
        for item in node:
            yield from values(item, key)
    elif not key:
        if node is not None:
            yield node
    elif isinstance(node, dict):
        yield from values(node.get(key[0]), key[1:])


def texts(row, fields):
    for field in fields:
        if field == "@mail":
            subject, body = row.get("subject"), row.get("body")
            if isinstance(subject, str) and isinstance(body, str):
                yield f"{subject}\n{body}"
        else:
            yield from (
                value for value in values(row, field.split(".")) if isinstance(value, str) and value.strip()
            )


def fingerprint(text):
    normalized = " ".join(unicodedata.normalize("NFKC", text).casefold().split())
    return hashlib.sha256(normalized.encode("utf-8", errors="surrogatepass")).digest()


def kind(value):
    return {str: "text", list: "list", dict: "object", bool: "bool", int: "int", float: "float"}.get(
        type(value), "other"
    )


def read_json(path):
    return json.loads(path.read_text(encoding="utf-8"))


def read_records(path):
    if path.name.endswith(".jsonl.bz2"):
        with bz2.open(path, "rt", encoding="utf-8") as stream:
            for line in stream:
                if line.strip():
                    yield json.loads(line)
    elif path.suffix == ".tsv":
        with path.open(encoding="utf-8-sig", newline="") as stream:
            yield from csv.DictReader(stream, delimiter="\t")
    else:
        yield from records(path)


def verify_file(file):
    path = ROOT / file["path"]
    if "digest" not in file:
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    if path.stat().st_size != file["bytes"]:
        raise ValueError(f"Sai kích thước: {file['path']}")
    algorithm = file["digest_type"]
    digest = hashlib.sha256() if algorithm == "sha256" else hashlib.sha1(usedforsecurity=False)
    if algorithm == "git-blob-sha1":
        digest.update(f"blob {file['bytes']}\0".encode())
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != file["digest"]:
        raise ValueError(f"Sai hash: {file['path']}")
    return path


def records_for(file, source):
    path = verify_file(file)
    if source != "phoatis":
        yield from read_records(path)
        return
    paths = [verify_file(item) for item in file["aligned"]]
    with (
        path.open(encoding="utf-8") as queries,
        paths[0].open(encoding="utf-8") as intents,
        paths[1].open(encoding="utf-8") as tags,
    ):
        for query, intent, tag in zip(queries, intents, tags, strict=True):
            if len(query.split()) != len(tag.split()):
                raise ValueError("Lệch số token và tag PhoATIS")
            yield {"seq.in": query.rstrip("\r\n"), "intent": intent.strip(), "slots": tag.strip()}


def locked_index():
    manifest = read_json(AUDIT / "eda_sources.json")
    found, missing = defaultdict(set), []
    for name, relative, _, cols, _ in manifest["benchmark"]:
        path = ROOT / relative
        if not path.is_file():
            missing.append(name)
            continue
        for row in records(path):
            if name == "CyberSecEval VI" and row.get("speaking_language") != "Vietnamese":
                continue
            for value in texts(row, cols):
                found[fingerprint(value)].add(name)
    return found, missing


def row_texts(row, file, spec):
    columns = file.get("text", spec["text"])
    if spec["id"] == "phoatis":
        return [row["seq.in"]]
    if spec["id"] == "vi_alpaca":
        return ["\n".join(v for v in (row.get("instruction"), row.get("input")) if v)]
    if spec["id"] == "5cd":
        return [
            item["value"]
            for item in row.get("conversations", [])
            if item.get("from") in {"human", "user"} and isinstance(item.get("value"), str)
        ]
    if spec["id"] == "oasst2" and row.get("role") != "prompter":
        return []
    return list(texts(row, columns))


def subset(row, spec, file):
    """Target strata for measuring coverage. Does not assign FOX labels."""
    source = spec["id"]
    if source == "oasst2":
        return (
            row.get("role") == "prompter"
            and row.get("lang") in {"vi", "en"}
            and row.get("deleted") is False
            and row.get("review_result") is True
            and row.get("synthetic") is False
        )
    if source == "aya":
        return row.get("language_code") in {"vie", "eng"}
    if source == "nailyk":
        return row.get("language") in {"vi", "en"}
    if source == "tensor_trust":
        return file["split"] == "attacks" and row.get("is_self_attack") is False
    return True


CORE = ("secai", "vi_en", "guardspec")
UNLABELLED = ("aya", "oasst2")
RESERVE = ("trustairlab", "deepset", "jackhhao")
SOURCES = CORE + UNLABELLED + RESERVE
LANGUAGE_MAP = {"Vietnamese": "vi", "English": "en", "Mixed": "mixed", "vie": "vi", "eng": "en"}
REVIEW_TYPES = {"Refusal Bypass", "Steganography", "Steganography (Acrostic)"}
CONTEXT_TAGS = {"indirect_injection", "tool_poisoning"}
DIRECT_TAGS = {"direct_injection", "jailbreak", "prompt_extraction"}


def interpret(source, row):
    """Use source labels to route rows for review, without producing final labels."""
    if source == "secai":
        return ("candidate", row["injection"]) if row["injection"] in {0, 1} else ("review_label", None)
    if source == "vi_en":
        if row["label"] == "BENIGN":
            return "candidate", 0
        if row["label"] == "INJECTION" and row["attack_type"] not in REVIEW_TYPES:
            return "candidate", 1
        return "review_label", None
    if source == "guardspec":
        tags = set(row["labels"])
        if tags & CONTEXT_TAGS:
            return "context", None
        if tags == {"benign"}:
            return "candidate", 0
        if tags & DIRECT_TAGS and "benign" not in tags:
            return "candidate", 1
        return "review_label", None
    return ("needs_annotation" if source in UNLABELLED else "reserve"), None


def guess_language(text):
    if len(re.findall(r"[^\W\d_]+", text)) < 5:
        return "uncertain"
    try:
        top = detect_langs(text[:1500])[0]
    except LangDetectException:
        return "uncertain"
    if top.prob < 0.90:
        return "uncertain"
    return top.lang if top.lang in {"vi", "en"} else "other"


def folded(text):
    text = unicodedata.normalize("NFKD", text.casefold().replace("đ", "d"))
    return " ".join("".join(c for c in text if not unicodedata.combining(c)).split())


def length_table(frame):
    def q(series, percent):
        return int(series.quantile(percent, interpolation="higher"))

    return (
        frame.groupby(["source", "language"], dropna=False)
        .agg(
            rows=("tokens", "size"),
            p50_tokens=("tokens", lambda x: q(x, 0.50)),
            p95_tokens=("tokens", lambda x: q(x, 0.95)),
            max_tokens=("tokens", "max"),
            over_512=("tokens", lambda x: x.gt(512).sum()),
            pii_hint=("pii", "sum"),
            secret_hint=("secret", "sum"),
            markup_hint=("markup", "sum"),
        )
        .reset_index()
    )


def load_candidates():
    specs = read_json(AUDIT / "sources.json")
    checks, rows, schema = [], [], []
    for source in SOURCES:
        spec = specs[source]
        for file in spec["files"]:
            split = file["split"]
            nontrain = split in {"test", "val", "validation"}
            counts = Counter()
            kinds, present, blank = {}, Counter(), Counter()
            for index, row in enumerate(records_for(file, source), 1):
                counts["raw"] += 1
                for column, value in row.items():
                    kinds.setdefault(column, set()).add(kind(value))
                    present[column] += value is not None
                    blank[column] += isinstance(value, str) and not value.strip()
                selected = subset(row, spec, file)
                if not selected:
                    continue
                counts["source_filter"] += 1
                for text in row_texts(row, file, spec):
                    if not text.strip():
                        continue
                    counts["with_text"] += 1
                    raw_lang = row.get(spec["language"])
                    lang = LANGUAGE_MAP.get(raw_lang, raw_lang) or "unknown"
                    group = next((row.get(k) for k in spec["group"] if row.get(k)), None)
                    route, target = interpret(source, row)
                    technique = row.get("attack_type", row.get("labels", ""))
                    if isinstance(technique, list):
                        technique = " + ".join(sorted(technique))
                    upstream = row.get("source", "") if source in {"guardspec", "trustairlab"} else ""
                    row_license = row.get("license", spec["license"])
                    digest = fingerprint(text).hex()
                    rows.append(
                        {
                            "source": source,
                            "split": split,
                            "file": file["path"],
                            "heldout": nontrain,
                            "line": index,
                            "text": text,
                            "digest": digest,
                            "language": lang,
                            "group": f"{source}:{group}" if group else "",
                            "route": route,
                            "target": target,
                            "technique": technique,
                            "variant": row.get("variant", ""),
                            "upstream": upstream,
                            "license": row_license,
                            "original_label": str(row.get(spec["label"], "")),
                        }
                    )
            checks.append(
                {
                    "source": source,
                    "split": split,
                    "file": file["path"],
                    "digest_type": file["digest_type"],
                    "digest": file["digest"],
                    **counts,
                }
            )
            schema.extend(
                {
                    "source": source,
                    "split": split,
                    "field": column,
                    "types": ", ".join(sorted(types)),
                    "non_null": present[column],
                    "missing_or_null": counts["raw"] - present[column],
                    "blank": blank[column],
                }
                for column, types in sorted(kinds.items())
            )
    frame = pd.DataFrame(rows)
    frame["target"] = frame["target"].astype("Int64")
    # Language estimation is bounded to a reproducible audit sample; remaining reserve rows stay unknown.
    DetectorFactory.seed = 0
    language_rows = []
    for source, bucket in frame.groupby("source", sort=False):
        if source in RESERVE:
            chosen = bucket.sample(n=min(180, len(bucket)), random_state=15)
        else:
            chosen = bucket.sample(n=min(100, len(bucket)), random_state=15)
        inferred = chosen["text"].map(guess_language)
        for (given, estimated), count in (
            chosen.assign(estimated=inferred).groupby(["language", "estimated"]).size().items()
        ):
            language_rows.append(
                {
                    "source": source,
                    "source_language": given,
                    "estimated_language": estimated,
                    "sample_rows": count,
                }
            )
    tokenizer_path = ROOT / "models/meta-llama-Llama-Prompt-Guard-2-86M/tokenizer.json"
    tokenizer = Tokenizer.from_file(str(tokenizer_path))
    tokenizer.no_truncation()
    tokenizer.no_padding()
    lengths = []
    strings = frame["text"].tolist()
    for i in range(0, len(strings), 256):
        lengths.extend(len(e.ids) for e in tokenizer.encode_batch(strings[i : i + 256]))
    frame["tokens"] = lengths
    frame["pii"] = frame["text"].map(lambda s: bool(PII_PATTERN.search(s)))
    frame["secret"] = frame["text"].map(lambda s: bool(SECRET_PATTERN.search(s)))
    frame["markup"] = frame["text"].map(lambda s: bool(MARKUP_PATTERN.search(s)))
    return frame, pd.DataFrame(checks).fillna(0), pd.DataFrame(language_rows), pd.DataFrame(schema)

In [2]:
def exact_audit(frame):
    refs, missing = locked_index()
    refs = {key.hex(): value for key, value in refs.items()}
    hits = [
        (source, split, name)
        for source, split, digest in frame[["source", "split", "digest"]].itertuples(index=False, name=None)
        for name in sorted(refs.get(digest, ()))
    ]
    matches = pd.DataFrame(hits, columns=["source", "split", "benchmark"])
    matches = matches.value_counts().rename("rows").reset_index()
    # Link exact duplicates and declared groups, including chains across publishers.
    parents = list(range(len(frame)))

    def root(index):
        while parents[index] != index:
            parents[index] = parents[parents[index]]
            index = parents[index]
        return index

    first_group, first_digest = {}, {}
    for index, (digest, group) in enumerate(frame[["digest", "group"]].itertuples(index=False, name=None)):
        for seen, key in ((first_digest, digest), (first_group, group)):
            if key:
                saved = seen.setdefault(key, index)
                parents[root(index)] = root(saved)
    frame = frame.assign(component=[root(i) for i in range(len(frame))])
    frame["benchmark_match"] = frame["digest"].isin(refs)
    potentially_labelled = frame[frame["route"].eq("candidate")]
    conflicts = potentially_labelled.groupby("component")["target"].nunique().loc[lambda s: s.gt(1)]
    # A group can legitimately contain both labels, e.g. near-minimal pairs.
    # Reject only conflicts on the same exact text.
    conflicting_texts = potentially_labelled.groupby("digest")["target"].nunique().loc[lambda s: s.gt(1)]
    protected = frame.loc[frame["heldout"] | frame["benchmark_match"], "component"].unique()
    frame["protected_group"] = frame["component"].isin(protected)
    frame["exact_label_conflict"] = frame["digest"].isin(conflicting_texts.index)
    return frame, matches, missing, len(conflicts), len(conflicting_texts)


def pair_overlap(frame):
    sets = {source: set(bucket["digest"]) for source, bucket in frame.groupby("source", sort=False)}
    summary = [
        {"source_a": a, "source_b": b, "exact_unique_texts": len(sets[a] & sets[b])}
        for a, b in combinations(sets, 2)
        if sets[a] & sets[b]
    ]
    return pd.DataFrame(summary).sort_values("exact_unique_texts", ascending=False)


def fold_overlap(frame):
    pool = frame.loc[~frame["protected_group"] & ~frame["heldout"] & frame["source"].isin(CORE + UNLABELLED)]
    heldout = frame.loc[frame["heldout"]]
    folded_heldout = {folded(text) for text in heldout["text"]}
    flags = pool["text"].map(folded).isin(folded_heldout)
    return pool.loc[flags].groupby("source").size().rename("additional_rows").reset_index()


def near_audit(frame):
    """Rank remaining train candidates against sampled upstream and FOX holdouts."""
    base = frame.loc[
        ~frame["protected_group"]
        & frame["route"].isin(["candidate", "needs_annotation"])
        & frame["text"].str.len().ge(32)
    ].drop_duplicates("digest")
    ref = frame.loc[frame["heldout"] & frame["text"].str.len().ge(32)].drop_duplicates("digest")
    samples = [
        part.sample(min(100, len(part)), random_state=15) for _, part in ref.groupby(["source", "split"])
    ]
    for name, relative, _, columns, _ in read_json(AUDIT / "eda_sources.json")["benchmark"]:
        candidates = []
        for i, row in enumerate(records(ROOT / relative), 1):
            if name == "CyberSecEval VI" and row.get("speaking_language") != "Vietnamese":
                continue
            candidates.extend(
                {"source": name, "split": "FOX benchmark", "line": i, "text": text}
                for text in texts(row, columns)
                if len(text) >= 32
            )
        if candidates:
            sample = pd.DataFrame(candidates).drop_duplicates("text")
            samples.append(sample.sample(min(60, len(sample)), random_state=15))
    ref = pd.concat(samples, ignore_index=True)
    vectorizer = HashingVectorizer(
        analyzer="char", ngram_range=(4, 4), n_features=2**19, alternate_sign=False, binary=True, norm="l2"
    )
    index = vectorizer.transform(base["text"])
    queries = vectorizer.transform(ref["text"])
    counts, examples = [], []
    for start in range(0, len(ref), 32):
        sims = (queries[start : start + 32] @ index.T).tocsr()
        for step in range(sims.shape[0]):
            q = ref.iloc[start + step]
            scores = sims.data[sims.indptr[step] : sims.indptr[step + 1]]
            docs = sims.indices[sims.indptr[step] : sims.indptr[step + 1]]
            for threshold in (0.85, 0.92):
                counts.append((q.source, q.split, threshold, int((scores >= threshold).any())))
            if len(scores) and scores.max() >= 0.85:
                top = scores.argmax()
                a = base.iloc[int(docs[top])]
                examples.append(
                    {
                        "source_train": a.source,
                        "row_train": a.line,
                        "source_ref": q.source,
                        "split_ref": q.split,
                        "row_ref": q.line,
                        "cosine": round(float(scores[top]), 3),
                    }
                )
    table = pd.DataFrame(counts, columns=["source", "split", "threshold", "has_match"])
    table = (
        table.groupby(["source", "split", "threshold"])
        .agg(checked=("has_match", "size"), flagged=("has_match", "sum"))
        .reset_index()
    )
    examples = pd.DataFrame(examples).sort_values("cosine", ascending=False) if examples else pd.DataFrame()
    return table, examples, len(base), len(ref)


def train_accounting(frame):
    eligible = frame.loc[~frame["heldout"] & frame["route"].isin(["candidate", "needs_annotation"])].copy()
    eligible["protection"] = np.select(
        [eligible["protected_group"], eligible["exact_label_conflict"]],
        ["benchmark or holdout group", "exact label conflict"],
        default="available for review",
    )
    funnel = (
        eligible.groupby(["source", "protection"])
        .agg(rows=("digest", "size"), unique_texts=("digest", "nunique"), components=("component", "nunique"))
        .reset_index()
    )
    retained = eligible.loc[eligible["protection"].eq("available for review")]
    coverage = (
        retained.groupby(["source", "language", "route", "target"], dropna=False)
        .agg(rows=("digest", "size"), unique_texts=("digest", "nunique"), components=("component", "nunique"))
        .reset_index()
    )
    return funnel, coverage


def report_inputs(frame, originals=False):
    priority = frame.loc[
        ~frame["heldout"]
        & ~frame["protected_group"]
        & ~frame["exact_label_conflict"]
        & frame["route"].eq("candidate")
    ]
    if originals:
        priority = priority.loc[priority["source"].ne("guardspec") | priority["variant"].eq("orig")]
    priority = priority.drop_duplicates("digest")
    coverage = priority.groupby(["language", "target"]).agg(
        texts=("digest", "nunique"), groups=("component", "nunique")
    )
    rows = []
    for language, fraction in {"vi": 0.45, "en": 0.45, "mixed": 0.10}.items():
        for label in (0, 1):
            key = (language, label)
            count, groups = coverage.loc[key].tolist() if key in coverage.index else (0, 0)
            rows.append(
                {
                    "language": language,
                    "provisional_label": label,
                    "texts": count,
                    "groups": groups,
                    "10k_target": round(10000 * fraction / 2),
                    "10k_gap": max(round(10000 * fraction / 2) - count, 0),
                    "40k_target": round(40000 * fraction / 2),
                    "40k_gap": max(round(40000 * fraction / 2) - count, 0),
                }
            )
    return pd.DataFrame(rows)


def digest_tokenizer():
    path = ROOT / "models/meta-llama-Llama-Prompt-Guard-2-86M/tokenizer.json"
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()

In [3]:
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 18)
pd.set_option("display.max_colwidth", 100)
frame, files, language_check, schema = load_candidates()
manifest = read_json(AUDIT / "sources.json")
display(pd.DataFrame([
    {"source": key, "revision": manifest[key]["revision"], "license": manifest[key]["license"]}
    for key in SOURCES
]))
display(files)
print("Token PG2, SHA-256 tokenizer:", digest_tokenizer())
for source, fields in schema.groupby("source", sort=False):
    print(source)
    display(fields)


,source,revision,license
0,secai,aab353fcb7e33838fe8b06df5d8afa45b7a03e54,Apache-2.0 (card nguồn)
1,vi_en,2afeb49278455c0e17efbc76acd2581237fb6f27,Apache-2.0 (card nguồn)
2,guardspec,c81d88c991406e7029c9945a9350f0f58f6ea220,Apache-2.0 cho mẫu dự án; giữ license từng dòng
3,aya,f9ea04583f02a8f86404ff6c58bf75fe637df8a2,apache-2.0
4,oasst2,179dd21fc55192153d94adb0e0ce8f69e222bf75,apache-2.0
5,trustairlab,a10aab8eff1c73165a442d4464dce192bd28b9c5,MIT (thẻ nguồn)
6,deepset,4f61ecb038e9c3fb77e21034b22511b523772cdd,Apache-2.0
7,jackhhao,2f2ceeb39658696fd3f462403562b6eea5306287,Apache-2.0


,source,split,file,digest_type,digest,raw,source_filter,with_text
0,secai,train,data/raw/eda01_vi/secai/data/train.jsonl,git-blob-sha1,366b4e852075529d89f036a64f01744135a60799,1005,1005,1005
1,secai,test,data/raw/eda01_vi/secai/data/test.jsonl,git-blob-sha1,dc5e99b40365ba1f0e4887584b0f6536de54c749,383,383,383
2,vi_en,train,data/raw/eda01_vi/vi_en/train.jsonl,git-blob-sha1,7c6132d5809da9ca41ae2cc9dcc6a81d52d9fa16,800,800,800
3,vi_en,test,data/raw/eda01_vi/vi_en/test.jsonl,git-blob-sha1,71f6434c5094fb1bcfb6ac0db4464a060bad8219,200,200,200
4,guardspec,train,data/raw/eda01_vi/guardspec/train/data/train.jsonl,git-blob-sha1,31aa7805fcb317e423e8d40d37535e7c28f4ffaf,31522,31522,31522
5,guardspec,val,data/raw/eda01_vi/guardspec/train/data/val.jsonl,git-blob-sha1,9187d259d9bad8d0e9c86a5fbfaa8321d953aec4,5125,5125,5125
6,aya,test,data/raw/eda05/aya/data/test-00000-of-00001.parquet,sha256,52bbb79461bff9887c9987a5737d9a06f5ec6c78c08d770828c37fc182ce2cca,1750,250,250
7,aya,train,data/raw/eda05/aya/data/train-00000-of-00001.parquet,sha256,51baa85043b569ba117f41516b32d1b0e4e2647fb203a08933f98a556fb1fb06,202362,12620,12618
8,oasst2,train,data/raw/eda05/oasst2/data/train-00000-of-00001-88ba0162028a73fc.parquet,sha256,9638227b41b0a4fb69afa5d5c4b52e3fb6abfd56c9870d81bae32c9981e769b1,128575,21654,21654
9,oasst2,validation,data/raw/eda05/oasst2/data/validation-00000-of-00001-1deeef95c3248fe0.parquet,sha256,fd46c63babc0756909a957ee40975bcf5610d58c5d1ab5ff2c57c0f57be1f273,6599,1124,1124


Token PG2, SHA-256 tokenizer: 3e7e96867c2acdd575f0862c74822e05d1d15b93d9d9a4a2144b1ce83ae3339f
secai


,source,split,field,types,non_null,missing_or_null,blank
0,secai,train,group,text,1005,0,0
1,secai,train,injection,int,1005,0,0
2,secai,train,intent,text,1005,0,0
3,secai,train,lang,text,1005,0,0
4,secai,train,pii,int,1005,0,0
5,secai,train,split,text,1005,0,0
6,secai,train,text,text,1005,0,0
7,secai,test,group,text,383,0,0
8,secai,test,injection,int,383,0,0
9,secai,test,intent,text,383,0,0


vi_en


,source,split,field,types,non_null,missing_or_null,blank
14,vi_en,train,attack_type,text,800,0,0
15,vi_en,train,label,text,800,0,0
16,vi_en,train,primary_language,text,800,0,0
17,vi_en,train,text,text,800,0,0
18,vi_en,test,attack_type,text,200,0,0
19,vi_en,test,label,text,200,0,0
20,vi_en,test,primary_language,text,200,0,0
21,vi_en,test,text,text,200,0,0


... (8 outputs left out)

deepset


,source,split,field,types,non_null,missing_or_null,blank
101,deepset,train,label,int,546,0,0
102,deepset,train,text,text,546,0,0


jackhhao


,source,split,field,types,non_null,missing_or_null,blank
103,jackhhao,train,prompt,text,1044,0,0
104,jackhhao,train,type,text,1044,0,0


## Ngôn ngữ, nhãn và nguồn gốc

Nhãn ứng viên được ánh xạ bảo thủ. SecAI dùng cờ injection độc lập với PII/intent. VI–EN giữ Refusal Bypass
và hai nhóm Steganography để rà vì tên kỹ thuật hoặc nội dung có hại tự nó chưa chứng minh hành vi vượt chỉ dẫn.
GuardSpec tách indirect/tool poisoning trước, rồi xét tag benign riêng hoặc direct/jailbreak/prompt extraction.
Các tag còn lại cần đọc nội dung. Aya/OASST2 và nguồn dự phòng chưa được gán target.

Kiểm tra ngôn ngữ trên mẫu lấy ngẫu nhiên có seed cố định dùng langdetect. Nhãn nguồn vẫn là nguồn số liệu chính
trong các bảng. Bộ nhận diện đơn ngữ khó nhận đúng câu trộn, câu ngắn và text mã hóa. Chênh lệch chỉ là tín hiệu
để đưa vào hàng chờ rà, không thay thế nhãn ngôn ngữ.

In [4]:
display(frame.groupby(["source", "split", "route", "target"], dropna=False).size().rename("rows").reset_index())
display(frame.groupby(["source", "language", "route"], dropna=False).size().unstack(fill_value=0))
display(language_check.sort_values(["source", "source_language", "sample_rows"], ascending=[True, True, False]))

,source,split,route,target,rows
0,aya,test,needs_annotation,<NA>,250
1,aya,train,needs_annotation,<NA>,12618
2,deepset,train,reserve,<NA>,546
3,guardspec,train,candidate,0,20507
4,guardspec,train,candidate,1,6495
5,guardspec,train,context,<NA>,3182
6,guardspec,train,review_label,<NA>,1338
7,guardspec,val,candidate,0,3539
8,guardspec,val,candidate,1,947
9,guardspec,val,context,<NA>,452


route                 candidate  context  needs_annotation  reserve  \
source      language                                                  
aya         en                0        0              4192        0   
            vi                0        0              8676        0   
deepset     unknown           0        0                 0      546   
guardspec   en            13774     1277                 0        0   
            vi            17714     2357                 0        0   
jackhhao    unknown           0        0                 0     1044   
oasst2      en                0        0             22722        0   
            vi                0        0                56        0   
secai       en              584        0                 0        0   
            vi              804        0                 0        0   
trustairlab unknown           0        0                 0    15140   
vi_en       en              378        0                 0        0   
            mixed           109        0                 0        0   
            vi              365        0                 0        0   

route                 review_label  
source      language                
aya         en                   0  
            vi                   0  
deepset     unknown              0  
guardspec   en                 474  
            vi                1051  
jackhhao    unknown              0  
oasst2      en                   0  
            vi                   0  
secai       en                   0  
            vi                   0  
trustairlab unknown              0  
vi_en       en                  60  
            mixed                2  
            vi                  86

,source,source_language,estimated_language,sample_rows
19,aya,en,en,27
20,aya,en,uncertain,4
21,aya,vi,vi,69
28,deepset,unknown,en,83
29,deepset,unknown,other,62
30,deepset,unknown,uncertain,35
14,guardspec,en,en,40
15,guardspec,en,uncertain,5
18,guardspec,vi,vi,35
17,guardspec,vi,uncertain,18


In [5]:
g = frame.query("source == 'guardspec' and split == 'train'")
display(g.groupby(["language", "variant", "route"]).size().rename("rows").unstack(fill_value=0))
display(g.groupby(["upstream", "license"]).agg(rows=("digest", "size"), groups=("group", "nunique")).reset_index())
display(frame.query("source == 'vi_en'").groupby(["split", "technique", "route"]).size().rename("rows").reset_index())

route                        candidate  context  review_label
language variant                                             
en       leet                      559       71            27
         orig                    11292     1048           389
vi       leet                      803       95            53
         no_diacritics            4338      595           270
         orig                     5414      723           331
         partial_diacritics        190       82             0
         teencode                 4406      568           268

,upstream,license,rows,groups
0,Lakera/gandalf_ignore_instructions:train,MIT,677,677
1,OpenAssistant/oasst2:prompts,Apache-2.0,2365,2339
2,deepset/prompt-injections:train,Apache-2.0,219,217
3,guard-spec,Apache-2.0,9899,5501
4,guard-spec:hardneg,Apache-2.0,1942,1084
5,guard-spec:realistic-fable,Apache-2.0,2148,780
6,guard-spec:v2,Apache-2.0,4958,1544
7,guard-spec:v3,Apache-2.0,3915,2710
8,guard-spec:v4,Apache-2.0,3128,1177
9,jackhhao/jailbreak-classification:train,Apache-2.0,756,753


,split,technique,route,rows
0,test,Character / Token Masking,candidate,20
1,test,Encoding/Obfuscation,candidate,7
2,test,Goal Hijacking,candidate,27
3,test,None,candidate,100
4,test,Prompt Leaking,candidate,23
5,test,Refusal Bypass,review_label,19
6,test,Steganography,review_label,4
7,train,Character / Token Masking,candidate,52
8,train,Encoding/Obfuscation,candidate,25
9,train,Goal Hijacking,candidate,106


## Độ dài thực tế và mẫu ưu tiên duyệt

Đếm bằng tokenizer PG2 cục bộ, có token đặc biệt, tắt padding và truncation. Các ca vượt 512 token cần
policy cửa sổ khớp lúc chạy thật. Nếu chia text dài, cần biết cửa sổ nào chứa chỉ dẫn được chèn trước khi gán 1.
Cờ PII, secret và markup là tín hiệu regex trên toàn pool, có thể khớp nhầm.

In [6]:
display(length_table(frame))
for source in CORE:
    b = frame.query("source == @source and split == 'train'")
    priorities = b[b["route"].isin(["review_label", "context"]) | b["tokens"].gt(512) | b["secret"]]
    display(priorities.sample(min(5, len(priorities)), random_state=15)[[
        "source", "split", "line", "language", "original_label", "technique", "variant", "route", "tokens"
    ]])

,source,language,rows,p50_tokens,p95_tokens,max_tokens,over_512,pii_hint,secret_hint,markup_hint
0,aya,en,4192,15,58,720,9,15,0,0
1,aya,vi,8676,74,1117,5426,1360,27,0,0
2,deepset,unknown,546,17,80,1046,1,0,0,0
3,guardspec,en,15525,27,130,2963,202,424,0,1
4,guardspec,vi,21122,32,97,352,0,642,0,0
5,jackhhao,unknown,1044,167,988,2963,217,2,0,0
6,oasst2,en,22722,20,78,2299,46,6,0,2
7,oasst2,vi,56,21,96,189,0,0,0,0
8,secai,en,584,14,38,139,0,11,3,2
9,secai,vi,804,19,48,152,0,11,8,3


,source,split,line,language,original_label,technique,variant,route,tokens
625,secai,train,626,vi,0,,,candidate,44
633,secai,train,634,vi,0,,,candidate,45
624,secai,train,625,vi,0,,,candidate,44
655,secai,train,656,vi,0,,,candidate,43
695,secai,train,696,vi,0,,,candidate,41


,source,split,line,language,original_label,technique,variant,route,tokens
2011,vi_en,train,624,en,INJECTION,Refusal Bypass,,review_label,27
1706,vi_en,train,319,vi,INJECTION,Refusal Bypass,,review_label,25
2027,vi_en,train,640,en,INJECTION,Refusal Bypass,,review_label,36
2093,vi_en,train,706,en,INJECTION,Steganography,,review_label,42
2034,vi_en,train,647,vi,INJECTION,Refusal Bypass,,review_label,64


,source,split,line,language,original_label,technique,variant,route,tokens
31297,guardspec,train,28910,vi,"['data_exfiltration', 'indirect_injection']",data_exfiltration + indirect_injection,teencode,context,57
5780,guardspec,train,3393,en,"['data_exfiltration', 'tool_poisoning']",data_exfiltration + tool_poisoning,orig,context,63
33242,guardspec,train,30855,vi,['tool_poisoning'],tool_poisoning,orig,context,96
9869,guardspec,train,7482,en,['jailbreak'],jailbreak,orig,candidate,548
29945,guardspec,train,27558,vi,['data_exfiltration'],data_exfiltration,no_diacritics,review_label,31


## Trùng, mâu thuẫn nhãn và nguy cơ rò rỉ

Fingerprint chuẩn hóa NFKC, casefold và khoảng trắng. Nếu một nội dung hoặc group gốc nối với test/val hay
benchmark khóa, toàn thành phần liên thông bị cách ly trong phép tính ngân sách. Thành phần có thể nối qua
nguồn khác. Một group có thể chứa cặp âm/dương hợp lệ. Chỉ cùng fingerprint và khác target mới đánh dấu xung đột.

Giao fingerprint sau bỏ dấu là tín hiệu bổ sung. Kiểm tra gần trùng dùng cosine trên character 4-gram của
mẫu tối đa 100 text mỗi split upstream và 60 text mỗi benchmark của FOX, với hai ngưỡng để thấy độ nhạy. Không suy ra sạch rò rỉ
khi không khớp và không tự loại theo cosine. Bản dịch, paraphrase và template khác chữ cần đối chiếu lineage
hoặc rà riêng. Phép so benchmark chính xác sử dụng manifest EDA 01 hiện có.

In [7]:
pool, exact_benchmark, missing, mixed_components, conflict_texts = exact_audit(frame)
assert not missing, f"Thiếu benchmark khóa: {missing}"
display(exact_benchmark)
display(pair_overlap(pool))
display(pool.groupby(["source", "split"]).agg(
    rows=("digest", "size"), unique_texts=("digest", "nunique"),
    groups=("component", "nunique"), protected=("protected_group", "sum")
).reset_index())
print("Thành phần có cả hai nhãn ứng viên:", mixed_components)
print("Fingerprint có nhãn ứng viên xung đột:", conflict_texts)
display(fold_overlap(pool))

,source,split,benchmark,rows
0,oasst2,train,LLMail Phase 2,30
1,trustairlab,regular_2023_12_25,LLMail Phase 2,18
2,oasst2,validation,LLMail Phase 2,1
3,secai,train,LLMail Phase 2,1


,source_a,source_b,exact_unique_texts
11,guardspec,jackhhao,864
8,guardspec,oasst2,666
9,guardspec,trustairlab,452
18,trustairlab,jackhhao,447
10,guardspec,deepset,245
2,secai,oasst2,12
15,oasst2,trustairlab,10
5,vi_en,aya,7
7,guardspec,aya,6
12,aya,oasst2,6


,source,split,rows,unique_texts,groups,protected
0,aya,test,250,249,249,250
1,aya,train,12618,10870,10869,96
2,deepset,train,546,546,546,29
3,guardspec,train,31522,31393,18226,68
4,guardspec,val,5125,5098,2929,5125
5,jackhhao,train,1044,1031,1031,117
6,oasst2,train,21654,21287,4815,2001
7,oasst2,validation,1124,1120,269,1124
8,secai,test,383,383,94,383
9,secai,train,1005,1005,277,5


Thành phần có cả hai nhãn ứng viên: 58
Fingerprint có nhãn ứng viên xung đột: 0


,source,additional_rows
0,guardspec,1


In [8]:
near_table, near_examples, n_pool, n_ref = near_audit(pool)
print(f"So gần trùng {n_ref:,} lượt text heldout với {n_pool:,} text ứng viên duy nhất.")
display(near_table)
display(near_examples.head(15))

So gần trùng 1,268 lượt text heldout với 54,911 text ứng viên duy nhất.


,source,split,threshold,checked,flagged
0,BIPIA · dev,FOX benchmark,0.85,60,0
1,BIPIA · dev,FOX benchmark,0.92,60,0
2,BIPIA · test,FOX benchmark,0.85,60,0
3,BIPIA · test,FOX benchmark,0.92,60,0
4,CSKH VI,FOX benchmark,0.85,60,0
5,CSKH VI,FOX benchmark,0.92,60,0
6,CyberSecEval EN,FOX benchmark,0.85,60,0
7,CyberSecEval EN,FOX benchmark,0.92,60,0
8,CyberSecEval VI,FOX benchmark,0.85,48,0
9,CyberSecEval VI,FOX benchmark,0.92,48,0


,source_train,row_train,source_ref,split_ref,row_ref,cosine
36,guardspec,12991,guardspec,val,2066,0.975
18,guardspec,19002,guardspec,val,3260,0.973
1,aya,180778,aya,test,1553,0.972
16,guardspec,19371,guardspec,val,3208,0.970
9,guardspec,18190,guardspec,val,3040,0.966
5,guardspec,11564,guardspec,val,1885,0.959
32,guardspec,20496,guardspec,val,3445,0.954
15,guardspec,1475,guardspec,val,158,0.953
4,aya,119475,aya,test,601,0.951
8,guardspec,11672,guardspec,val,1913,0.946


## Số lượng còn lại sau cách ly

Các dòng Aya/OASST2 ở trạng thái needs_annotation chưa được tính vào ngân sách nhóm có nhãn. Các nguồn
dự phòng đợi kiểm nguồn gốc và nhãn ngôn ngữ. Dòng, text duy nhất và thành phần liên thông là các đơn vị
đếm khác nhau. Tổng text duy nhất theo từng nguồn có thể đếm lặp giữa nguồn.

Bảng ngân sách mô phỏng 10k và 40k với tỷ lệ 45/45/10 và giả định 0/1 cân bằng trong từng ngôn ngữ.
Tính cả biến thể hợp lệ ở bảng đầu và chỉ giữ variant=orig của GuardSpec ở bảng tiếp theo để thấy
độ nhạy với việc tăng dữ liệu. Gap là phần thiếu theo từng ô, không bù giữa các ngôn ngữ hay nhãn.


In [9]:
funnel, coverage = train_accounting(pool)
budget = report_inputs(pool)
display(funnel)
display(coverage)
print("Tính cả biến thể GuardSpec")
display(budget)
print("Chỉ variant=orig của GuardSpec")
display(report_inputs(pool, originals=True))
print("Tổng raw, tổng qua lọc nguồn:", int(files["raw"].sum()), len(pool))


assert pool.loc[pool["route"].ne("candidate"), "target"].isna().all()
assert pool.loc[pool["route"].eq("candidate"), "target"].isin([0, 1]).all()
assert pool.loc[pool["heldout"] | pool["benchmark_match"], "protected_group"].all()
assert set(pool["source"]) == set(SOURCES)
print("Kiểm toàn vẹn ánh xạ và cách ly đã qua.")

,source,protection,rows,unique_texts,components
0,aya,available for review,12522,10801,10801
1,aya,benchmark or holdout group,96,69,68
2,guardspec,available for review,26936,26825,15963
3,guardspec,benchmark or holdout group,66,64,38
4,oasst2,available for review,19653,19582,4694
5,oasst2,benchmark or holdout group,2001,1705,121
6,secai,available for review,1000,1000,276
7,secai,benchmark or holdout group,5,5,1
8,vi_en,available for review,673,662,662
9,vi_en,benchmark or holdout group,2,2,1


,source,language,route,target,rows,unique_texts,components
0,aya,en,needs_annotation,<NA>,3846,3142,3142
1,aya,vi,needs_annotation,<NA>,8676,7659,7659
2,guardspec,en,candidate,0,8861,8856,8043
3,guardspec,en,candidate,1,2927,2916,2534
4,guardspec,vi,candidate,0,11580,11504,4107
5,guardspec,vi,candidate,1,3568,3551,1336
6,oasst2,en,needs_annotation,<NA>,19605,19534,4659
7,oasst2,vi,needs_annotation,<NA>,48,48,35
8,secai,en,candidate,0,290,290,78
9,secai,en,candidate,1,66,66,22


Tính cả biến thể GuardSpec


,language,provisional_label,texts,groups,10k_target,10k_gap,40k_target,40k_gap
0,vi,0,12264,4442,2250,0,9000,0
1,vi,1,3725,1468,2250,0,9000,5275
2,en,0,9325,8299,2250,0,9000,0
3,en,1,3102,2676,2250,0,9000,5898
4,mixed,0,18,18,500,482,2000,1982
5,mixed,1,51,51,500,449,2000,1949


Chỉ variant=orig của GuardSpec


,language,provisional_label,texts,groups,10k_target,10k_gap,40k_target,40k_gap
0,vi,0,4875,4065,2250,0,9000,4125
1,vi,1,1472,1342,2250,778,9000,7528
2,en,0,8887,8274,2250,0,9000,113
3,en,1,2981,2675,2250,0,9000,6019
4,mixed,0,18,18,500,482,2000,1982
5,mixed,1,51,51,500,449,2000,1949


Tổng raw, tổng qua lọc nguồn: 395051 91411
Kiểm toàn vẹn ánh xạ và cách ly đã qua.


## Bàn giao sang ELT

1. **Extract.** Dùng revision và hash từ manifest 0.5. Truy lại dòng qua nguồn, tệp, split và chỉ số dòng.
2. **Load.** Bảng staging giữ text gốc, nhãn gốc, ngôn ngữ, ID nhóm, loại biến thể, nguồn upstream và license.
   Giá trị chưa gán nhãn nằm trong hàng chờ. Tách quyền dùng và trạng thái duyệt thành các cột riêng.
3. **Transform.** Chốt taxonomy 0/1. Duyệt các nhóm có dấu hiệu nhiễu nhãn, tiếng Việt và câu trộn.
   Chọn một bản đại diện cho exact duplicate, giữ provenance, giải quyết xung đột trước khi chia.
   Nối bản dịch, biến thể, template và hội thoại vào cùng group. Cách ly benchmark và giữ định danh split gốc.
4. **Tạo tập train.** Dùng text và label 0/1. Giữ metadata audit ngoài input model. Kiểm coverage theo
   ngôn ngữ × nhãn × họ attack và nguồn. Dùng dev để chọn ngưỡng. TMĐT giữ vai trò lát cắt đánh giá.

Cần rà nhãn có phân tầng cho mẫu khó và tự nhiên. Với mẫu mơ hồ khi đọc riêng user text, chuyển review/CONTEXT.
Quyết định về near-duplicate, quyền dùng upstream và quota cần được duyệt trước khi xuất dataset cuối.

| Trường staging đề xuất | Lấy từ đâu hoặc cách tính |
|---|---|
| source, revision, source_file, source_split, source_row_id | Manifest pin + khóa nguồn hoặc chỉ số dòng |
| text_raw, source_label, source_language, upstream_source, license | Giá trị gốc. Provenance và license riêng từng dòng nếu có |
| label_status, label, language, entry_point | Duyệt theo rubric. Train yêu cầu label đã duyệt thuộc 0/1 |
| origin_group_id, variant, exact_hash, duplicate_group_id | Nhóm nguồn, hội thoại, bản dịch, cụm trùng được xác nhận |
| token_count, benchmark_match, split | Tokenizer PG2, đối chiếu benchmark, chia theo nhóm |

Điều kiện chuyển sang train là text có nguồn gốc và quyền dùng rõ, nhãn hợp lệ, nhóm không giao split,
tỷ lệ thực tế được ghi lại theo ngôn ngữ × nhãn và dữ liệu chưa duyệt nằm ngoài export.
